# Create a ground truth data

source: https://www.youtube.com/watch?v=YScoH28cVf8&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv



### get the llm-zoomcamp data

In [ ]:

import sys
import os
import json
from rich import print
parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import load_faq_data

documents = load_faq_data()

# we will create ground truth data for the llm-zoomcamp  to get a small data set
documents = [doc for doc in documents if doc['course'] == 'llm-zoomcamp']

# check the length
print(f'The number of documents for the llm-zoomcamp is : {len(documents)}')
print(documents[0])

# the dictionary (document) is converted to string
user_prompt = json.dumps(documents[0])
print(f'The user prompt is: {user_prompt}')

The number of documents for the llm-zoomcamp is : 103

{
    'id': '74eb249bbf',
    'course': 'llm-zoomcamp',
    'section': 'General Course-Related Questions',
    'question': 'I just discovered the course. Can I still join?',
    'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still 
accepting submissions.'
}

The user prompt is: {"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", 
"question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a 
certificate, you need to submit your project while we\u2019re still accepting submissions."}

In [11]:
type(user_prompt)

str

### create questions using llm for each document

In [35]:

# instead of generating text using llm, I tell llm to get response as I wish, a dictionary with keys
from pydantic import BaseModel 

class Questions(BaseModel):
    """
        I want the model llm to return sth like this:: { "questions": [..., ..., ]}
        a key questions that has list value, each list has n elements
    """
    questions: list[str]


# instructions tell the model:

# pretend to be a student
# read an FAQ
# generate 5 possible questions
# don't copy too much wording
# sound natura
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

#   Using openAI
# openai_client = OpenAI()
# messages = [
#     {"role": "developer", "content": data_gen_instructions},
#     {"role": "user", "content": user_prompt}
# ]

# # note parse to get response.questions instead of response['questions']
# response = openai_client.responses.parse(
#     model="gpt-5.4-mini",
#     input=messages,
#     text_format=Questions
# )
# result = response.output_parsed

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
import textwrap
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content=data_gen_instructions),
    HumanMessage(content=user_prompt)
]
llm_model = ChatOllama(
            model="gemma3:4b",
            base_url="http://localhost:11434",
            temperature = 0,
        )
structured_llm = llm_model.with_structured_output(Questions, 
                                                  include_raw=True, # to get the metadata including the input and output tokens
                                                  )
result = structured_llm.invoke(messages)
print(result)




{
    'raw': AIMessage(
        content='{\n  "questions": [\n    "Hi! I’m really interested in this LLM Zoomcamp. I just found it now – is
there any chance I can still sign up, even though the course has already started?",\n    "I saw that you need a 
project to get a certificate. What exactly does ‘while we’re still accepting submissions’ mean? Is there a deadline
for when I have to finish and submit my project?",\n    "Okay, so if I *do* join now, will I be able to get a 
certificate? It says I only get one if I submit a project during the time you\'re accepting them – can you explain 
that a little more?",\n    "Just wondering - if I’m not planning on getting a certificate, can I still participate 
in the Zoomcamp and complete the course materials?",\n    "I read that to get a certificate, I need to submit my 
project while submissions are accepted. What kind of projects are you looking for? Are there any specific 
requirements?"\n  ]\n}',
        additional_kwargs={},
        response_metadata={
            'model': 'gemma3:4b',
            'created_at': '2026-07-07T12:48:57.1867388Z',
            'done': True,
            'done_reason': 'stop',
            'total_duration': 11685505200,
            'load_duration': 6699776400,
            'prompt_eval_count': 188,
            'prompt_eval_duration': 263432000,
            'eval_count': 217,
            'eval_duration': 4711233000,
            'logprobs': None,
            'model_name': 'gemma3:4b',
            'model_provider': 'ollama'
        },
        id='lc_run--019f3c9f-adbb-73b3-b647-da45c17d7f92-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={'input_tokens': 188, 'output_tokens': 217, 'total_tokens': 405}
    ),
    'parsed': Questions(
        questions=[
            'Hi! I’m really interested in this LLM Zoomcamp. I just found it now – is there any chance I can still 
sign up, even though the course has already started?',
            'I saw that you need a project to get a certificate. What exactly does ‘while we’re still accepting 
submissions’ mean? Is there a deadline for when I have to finish and submit my project?',
            "Okay, so if I *do* join now, will I be able to get a certificate? It says I only get one if I submit a
project during the time you're accepting them – can you explain that a little more?",
            'Just wondering - if I’m not planning on getting a certificate, can I still participate in the Zoomcamp
and complete the course materials?',
            'I read that to get a certificate, I need to submit my project while submissions are accepted. What 
kind of projects are you looking for? Are there any specific requirements?'
        ]
    ),
    'parsing_error': None
}

In [38]:
print(result['parsed'].questions)

[
    'Hi! I’m really interested in this LLM Zoomcamp. I just found it now – is there any chance I can still sign up,
even though the course has already started?',
    'I saw that you need a project to get a certificate. What exactly does ‘while we’re still accepting 
submissions’ mean? Is there a deadline for when I have to finish and submit my project?',
    "Okay, so if I *do* join now, will I be able to get a certificate? It says I only get one if I submit a project
during the time you're accepting them – can you explain that a little more?",
    'Just wondering - if I’m not planning on getting a certificate, can I still participate in the Zoomcamp and 
complete the course materials?',
    'I read that to get a certificate, I need to submit my project while submissions are accepted. What kind of 
projects are you looking for? Are there any specific requirements?'
]

In [41]:
print(documents[0]['answer'])

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting 
submissions.

# put it in function

In [ ]:

class Questions(BaseModel):
    """
        I want the model llm to return sth like this: { "questions": [..., ..., ]}
        a key questions that has list value, each list has n elements
    """
    questions: list[str]



def llm_structured(instructions, user_prompt, output_type=Questions, model="gemma3:4b"):


    messages = [
    SystemMessage(content=instructions),
    HumanMessage(content=user_prompt)
    ]

    llm_model = ChatOllama(
            model=model,
            base_url="http://localhost:11434",
            temperature = 0,
        )
    structured_llm_model = llm_model.with_structured_output(
                                        output_type, 
                                        include_raw=True, # to get the metadata including the input and output tokens
    )
    result = structured_llm_model.invoke(messages)

    return result['parsed'].questions, result['raw'].usage_metadata

response, token = llm_structured(instructions=data_gen_instructions, user_prompt=user_prompt, output_type=Questions, model="gemma3:4b")